# P3 First results on new data: Min-K% Prob on WikiMIA-2026

**T4 GPU**, then **Run all** (about 10–15 minutes).

WikiMIA-2026 is built by `scripts/build_wikimia2026.py` in this repository, following the paper's own recipe
(Section 2.2) with a **1 January 2026** cut-off. It is written in the same JSONL schema the original code reads,
so the original `run.py` functions score it **unmodified**.

In [ ]:
# Clone the ORIGINAL implementation at the exact commit used throughout the project.
%cd -q /content
!git clone --quiet https://github.com/swj0419/detect-pretrain-code.git 2>/dev/null || echo 'already cloned'
!cd /content/detect-pretrain-code && git checkout --quiet a5048f93c4fa6491aab0638bb3e6fd5a662d7698 && git log -1 --format='original code @ %h (%ad)' --date=short
# run.py needs accelerate (device_map='auto') and imports openai unconditionally.
!pip install -q accelerate openai

import json, os, platform, sys
import torch, transformers, datasets, sklearn, numpy as np
ENV = {'python': platform.python_version(), 'torch': torch.__version__,
       'transformers': transformers.__version__, 'datasets': datasets.__version__,
       'sklearn': sklearn.__version__, 'numpy': np.__version__,
       'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}
print(json.dumps(ENV, indent=1))
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU, then Run all again.'

In [ ]:
# Import the original modules and call THEIR functions. Nothing in run.py / eval.py is edited;
# we only drive them from Python so the per-passage scores can be saved (run.py discards them).
SRC = '/content/detect-pretrain-code/src'
os.chdir(SRC); sys.path.insert(0, SRC)
import run as R            # original run.py  (load_model, evaluate_data, ...)
import gc, re, time
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

REF = 'EleutherAI/pythia-70m'   # the paper's reference model for Pythia-2.8B (Section 4.2)

def metric_name(key):
    """Map run.py's prediction keys to the paper's method names."""
    if key == 'ppl': return 'PPL'
    if key.startswith('ppl/Ref'): return 'Smaller Ref'
    if key.startswith('ppl/lowercase'): return 'Lowercase'
    if key.startswith('ppl/zlib'): return 'Zlib'
    m = re.match(r'Min_([0-9.]+)% Prob', key)          # note: run.py writes 'Min_30.000000000000004% Prob'
    return f'Min-{round(float(m.group(1)))}%' if m else key

BOOT_METRICS = {'PPL', 'Zlib', 'Lowercase', 'Smaller Ref', 'Min-20%'}

def summarise(outputs, model, dataset, length, n_boot=1000, seed=0):
    """AUC + TPR@5%FPR per method using the ORIGINAL eval.sweep, plus a bootstrap 95% CI on AUC."""
    labels = np.array([ex['label'] for ex in outputs], dtype=bool)
    rng = np.random.default_rng(seed)
    idx = [rng.integers(0, len(labels), len(labels)) for _ in range(n_boot)]
    rows = []
    for key in outputs[0]['pred']:
        scores = np.array([ex['pred'][key] for ex in outputs], dtype=float)
        if np.isnan(scores).any():
            continue
        fpr, tpr, auc, acc = R.sweep(scores, labels)          # original: roc_curve(labels, -score)
        tpr5 = tpr[np.where(fpr < .05)[0][-1]]
        lo = hi = np.nan
        name = metric_name(key)
        if name in BOOT_METRICS:
            boots = [roc_auc_score(labels[i], -scores[i]) for i in idx if 0 < labels[i].sum() < len(i)]
            lo, hi = np.percentile(boots, [2.5, 97.5])
        rows.append(dict(dataset=dataset, model=model, length=length, n=len(labels), method=name,
                         auc=auc, auc_lo=lo, auc_hi=hi, tpr_at_5fpr=tpr5, best_acc=acc))
    return rows

def save_jsonl(rows, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as f:
        for r in rows:
            f.write(json.dumps({'input': r['input'], 'label': int(r['label']), 'pred': r['pred']}) + '\n')

def run_model(target, jobs, out_root, dataset_name):
    """jobs: list of (length, list_of_examples). Loads the model once, scores every job."""
    rows = []
    todo = [(L, data) for L, data in jobs
            if not os.path.exists(f'{out_root}/scores/{target.split("/")[-1]}_len{L}.jsonl')]
    for L, data in jobs:                                  # resume: reuse finished runs
        p = f'{out_root}/scores/{target.split("/")[-1]}_len{L}.jsonl'
        if os.path.exists(p):
            done = [json.loads(l) for l in open(p)]
            rows += summarise(done, target, dataset_name, L)
    if not todo:
        return rows
    t0 = time.time()
    m1, m2, t1, t2 = R.load_model(target, REF)             # original loader
    print(f'{target}: loaded in {time.time()-t0:.0f}s, dtype={m1.dtype}')
    for L, data in todo:
        t0 = time.time()
        out = R.evaluate_data(data, m1, m2, t1, t2, 'input', target, REF)   # original scoring loop
        d = f'{out_root}/original_eval/{target}_{REF}/len{L}'
        os.makedirs(d, exist_ok=True)
        try:
            R.fig_fpr_tpr(out, d)                          # original auc.txt + auc.png
        except ValueError as e:                            # e.g. NaN from Min-5% on a very short passage
            print('  original eval.py failed:', e)
        plt.close('all')
        save_jsonl(out, f'{out_root}/scores/{target.split("/")[-1]}_len{L}.jsonl')
        rows += summarise(out, target, dataset_name, L)
        print(f'  length {L}: {len(data)} passages in {time.time()-t0:.0f}s')
    del m1, m2; gc.collect(); torch.cuda.empty_cache()
    return rows
print('driver ready')

In [ ]:
!git clone --quiet https://github.com/Ashik-here/COMP8240-MinK-Replication.git /content/mine 2>/dev/null || (cd /content/mine && git pull --quiet)
DATA = '/content/mine/data/wikimia2026'
LENGTHS = [32, 64, 128, 256]
new = {L: R.load_jsonl(f'{DATA}/WikiMIA2026_length{L}.jsonl') for L in LENGTHS}   # original loader
print({L: len(v) for L, v in new.items()})

MODELS = ['EleutherAI/pythia-160m', 'EleutherAI/pythia-1.4b', 'EleutherAI/pythia-2.8b']
OUT = '/content/results/wikimia2026'
rows = []
for target in MODELS:
    rows += run_model(target, [(L, new[L]) for L in LENGTHS], OUT, 'WikiMIA-2026')
    pd.DataFrame(rows).to_csv(f'{OUT}/summary.csv', index=False)
json.dump(ENV, open(f'{OUT}/environment.json', 'w'), indent=1)

df = pd.read_csv(f'{OUT}/summary.csv')
main = ['PPL', 'Zlib', 'Lowercase', 'Smaller Ref', 'Min-20%']
t = df[df.method.isin(main)].pivot_table(index=['model', 'method'], columns='length', values='auc')
t['mean'] = t.mean(axis=1); display(t.round(3))

In [ ]:
!cd /content && zip -qr wikimia2026_results.zip results/wikimia2026
from google.colab import files
files.download('/content/wikimia2026_results.zip')